# 🎬 Promo Extractor (Kuku · Problem 1)
Finds 20–60 s promo clips in a microdrama screenplay: a hook in the first 3 s, a cliffhanger at the cut, and clear to someone who has never seen the show.

**Manual LLM mode (no API key).** The judging is done by a real LLM, but you run it: the notebook packs the model's tasks into a few large prompts, you paste each into ChatGPT, Claude or Gemini in your browser, and paste the reply back. Python checks every answer and every timestamp, then prepares the next round (find hooks → judge clips → rank).

Run the cells in order.

## 1. Get the code

In [ ]:
REPO = "https://github.com/ShubNoob/promo-extractor"   # <- your repo

import glob, os, shutil
%cd /content
for z in glob.glob("/content/repo/**/*.zip", recursive=True):   # keep an uploaded scripts zip
    shutil.copy(z, "/content/")
!rm -rf repo && git clone -q $REPO repo
hits = glob.glob("/content/repo/**/manual.py", recursive=True)
assert hits, "manual.py not found - upload the latest files to GitHub"
%cd {os.path.dirname(hits[0])}
!pip -q install pydantic pyyaml ipywidgets

## 2. Episode scripts (.zip of .txt files)
Uses a scripts zip already in `/content`, otherwise asks you to upload it. Code zips are ignored.

In [ ]:
import glob, os, shutil, zipfile
from google.colab import files

def episode_count(path):
    """How many episode .txt scripts a zip holds (0 if it is not a scripts zip)."""
    try:
        names = zipfile.ZipFile(path).namelist()
    except (zipfile.BadZipFile, FileNotFoundError):
        return 0
    return sum(n.lower().endswith(".txt") and "episode" in os.path.basename(n).lower()
               and "__macosx" not in n.lower() for n in names)

zips = [z for z in glob.glob("/content/*.zip") if episode_count(z) >= 5]
if zips:
    ZIP = zips[0]
else:
    print("No scripts zip found. Upload the zip of episode .txt files (NOT the code zip):")
    name = next(iter(files.upload()))
    ZIP = shutil.copy(os.path.abspath(name), "/content/")   # keep it safe in /content
    assert episode_count(ZIP) >= 5, f"{name} has no episode scripts - wrong zip?"
print(f"Scripts: {ZIP} ({episode_count(ZIP)} episode files)")

## 3. Checks (17 automated tests)

In [ ]:
!python test_checks.py | tail -1

## 4. Settings
- `CHAT_MODEL`: the chat model you will paste into (it is written into the results).
- `HOOKS`: hooks per episode. 3 keeps the number of pastes down.
- `MAX_CHARS`: size of each prompt. 40,000 suits ChatGPT; Claude.ai takes 90,000+ (fewer pastes). Lower it if the chat says the message is too long.
- `EPISODES`: `""` for all, or e.g. `"1-10"` to try a few first.

In [ ]:
CHAT_MODEL = "ChatGPT (web)"      # e.g. "Claude Sonnet (claude.ai)", "Gemini (web)"
HOOKS = 3
MAX_CHARS = 40000
EPISODES = ""
MANUAL_DIR = "/content/manual_work"   # pasted answers live here, outside the repo

## 5. Paste rounds
For each batch: click in the **top box**, press **Ctrl+A** then **Ctrl+C**, paste it into a **new chat**, wait for the full reply, copy the **whole reply** into the **lower box**, and click **Save reply**. The tool checks it and moves to the next batch; when a round is done it prepares the next one by itself. At the end it shows the shortlist.

If a reply misses some tasks, they are asked again in the next round. You can stop any time and rerun this cell later: saved answers are kept (see section 6 to back them up).

In [ ]:
import importlib, manual
importlib.reload(manual)
manual.colab_ui(ZIP, out="outputs", manual_dir=MANUAL_DIR, label=CHAT_MODEL,
                hooks=HOOKS, episodes=EPISODES, max_chars=MAX_CHARS)

### 5b. Same thing without the widget (only if the boxes above do not appear)
Run the first cell to prepare a round, the second to see a batch, paste the reply into the third (under the first line) and run it, then the fourth to save it. Change `batch_01` to the next batch each time; when all are saved, run the first cell again for the next round.

In [ ]:
!python run.py "$ZIP" --provider manual --model "$CHAT_MODEL" --manual-dir $MANUAL_DIR --hooks $HOOKS --max-chars $MAX_CHARS --out outputs | tail -3
!python manual.py status --manual-dir $MANUAL_DIR

In [ ]:
!python manual.py show batch_01 --manual-dir $MANUAL_DIR

In [ ]:
%%writefile /content/reply.txt
PASTE THE MODEL'S WHOLE REPLY HERE, replacing this line

In [ ]:
!python manual.py answer batch_01 /content/reply.txt --manual-dir $MANUAL_DIR

## 6. Back up your answers (do this before closing Colab)
Colab forgets files when the session ends. Download the answers; to resume later, upload the zip in the second cell and rerun section 5.

In [ ]:
from google.colab import files
!cd /content && zip -qr manual_answers.zip manual_work
files.download("/content/manual_answers.zip")

In [ ]:
up = next(iter(files.upload()))          # restore: pick manual_answers.zip
!cd /content && unzip -qo "$up"
!python manual.py status --manual-dir $MANUAL_DIR

## 7. Results

In [ ]:
from IPython.display import Markdown, display
import json
log = json.load(open("outputs/run_log.json"))
print("Model:", log["model"], "| episodes:", log["episodes"], "| hooks:", log["hooks_found"],
      "| clips judged:", log["clips_judged"], "| tiers:", log["tier_counts"],
      "| rejected by checks:", log["rejected_before_judging"], "| end times corrected:", log["end_timestamps_adjusted"],
      "| still pending:", log.get("pending_manual_tasks", 0))
display(Markdown(open("outputs/shortlist.md").read()))

## 8. Download the raw outputs and the hand-check sheet

In [ ]:
!zip -qr raw_outputs.zip outputs
files.download("raw_outputs.zip")
files.download("outputs/hand_check.csv")

## 9. Hand check (human in the loop)
Open `hand_check.csv`, read each clip in its episode's script between `start_ts` and `end_ts`, fill `human_good` (y/n) and `human_reason`, save as CSV, then upload it here.

In [ ]:
labelled = next(iter(files.upload()))
!python run.py --score "$labelled"